# SSMD and Matching Analysis of CDPHE vs Highest r.grf Choices

In [1]:
import copy
import functools
import glob
import pickle
import scipy
import sklearn
import warnings

warnings.filterwarnings('ignore')

import itertools
import graphviz as gr
import numpy as np
import os
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm


from matplotlib import style
from matplotlib import pyplot as plt
from pprint import pprint
from scipy import stats, special

from sklearn import datasets, mixture
from sklearn.covariance import EmpiricalCovariance
from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import StandardScaler

from tqdm import tqdm

from urllib.request import urlopen
import json
with urlopen('https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json') as response:
    counties = json.load(response)

import plotly.express as px
import plotly.figure_factory as ff

%matplotlib inline

style.use("fivethirtyeight")
pd.set_option("display.max_columns", 6)

np.random.seed(0)
pd.set_option('display.max_columns', None)

### Set Directories

In [2]:
DATA_FOLDER = "../data"
FIGURES_FOLDER = "../figures"
OUTPUT_FOLDER = "../output"
NURSE_CORRECTION_FOLDER = os.path.join(DATA_FOLDER,"Case Study COVID County Matching Databases")

### Import County Fips master

In [3]:
county_fips_master_df = pd.read_csv(os.path.join(OUTPUT_FOLDER,"county_fips_master.csv"))
county_fips_master_df = county_fips_master_df[county_fips_master_df["state_abbr"]=="CO"]
county_fips_master_df.head()

,fips,county_name,state_abbr,state_name,long_name,sumlev,region,division,state,county,crosswalk,region_name,division_name
245,8001,Adams County,CO,Colorado,Adams County CO,50.0,4.0,8.0,8.0,1.0,4-8-8-1,West,Mountain
246,8003,Alamosa County,CO,Colorado,Alamosa County CO,50.0,4.0,8.0,8.0,3.0,4-8-8-3,West,Mountain
247,8005,Arapahoe County,CO,Colorado,Arapahoe County CO,50.0,4.0,8.0,8.0,5.0,4-8-8-5,West,Mountain
248,8007,Archuleta County,CO,Colorado,Archuleta County CO,50.0,4.0,8.0,8.0,7.0,4-8-8-7,West,Mountain
249,8009,Baca County,CO,Colorado,Baca County CO,50.0,4.0,8.0,8.0,9.0,4-8-8-9,West,Mountain


### Import Nursing Homes and Correctional Facilitiy Files

In [4]:
nursing_ts_df = pd.read_csv(os.path.join(NURSE_CORRECTION_FOLDER,"ReplicationData_TimeSeries_CO.csv"))
nursing_ts_df.head()

,Date,PROVNUM,PROVNAME,CITY,COUNTY,Case,ever_had_case,first_week,case_count_outbreak,case_count_CMS,Case_a,ever_had_case_a,first_week_a,homes_this,homes_last,homes_two,connections_this,connections_last,connections_two,homes_this_a,homes_last_a,homes_two_a,connections_this_a,connections_last_a,connections_two_a,OOS_homes,OOS_connections
0,19apr2020 00:00:00,065001,LOWRY HILLS CARE AND REHABILITATION,AURORA,Arapahoe County,1,1,19apr2020 00:00:00,4,0,1,1,19apr2020 00:00:00,1,0,0,2,0,0,1,0,0,2,0,0,0,0
1,19apr2020 00:00:00,065009,ST PAUL HEALTH CENTER,DENVER,Denver County,0,0,26apr2020 00:00:00,0,0,0,0,26apr2020 00:00:00,2,0,0,2,0,0,2,0,0,2,0,0,0,0
2,19apr2020 00:00:00,065015,MOUNTAIN VISTA HEALTH CENTER,WHEAT RIDGE,Jefferson County,1,1,19apr2020 00:00:00,5,0,1,1,19apr2020 00:00:00,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,19apr2020 00:00:00,065034,AMBERWOOD COURT REHABILITATION AND CARE COMMUNITY,DENVER,Denver County,1,1,19apr2020 00:00:00,10,0,1,1,19apr2020 00:00:00,4,0,0,9,0,0,4,0,0,9,0,0,0,0
4,19apr2020 00:00:00,065052,MESA VISTA OF BOULDER,BOULDER,Boulder County,1,1,19apr2020 00:00:00,15,0,1,1,19apr2020 00:00:00,1,0,0,1,0,0,1,0,0,1,0,0,1,1


In [5]:
nursing_cross_df = pd.read_csv(os.path.join(NURSE_CORRECTION_FOLDER,"ReplicationData_CrossSection.csv"))
nursing_cross_df = nursing_cross_df[nursing_cross_df["providerstate"]=="CO"].sort_values(by="Provider_ID")
nursing_cross_df.head()

,weekending,Provider_ID,providername,provideraddress,providercity,providerstate,providerzipcode,submitteddata,passedqualityassurancecheck,residentsweeklyadmissionscovid19,residentstotaladmissionscovid19,residentsweeklyconfirmedcovid19,CMS_ResidentsTotalConfirmed,residentsweeklysuspectedcovid19,residentstotalsuspectedcovid19,residentsweeklyalldeaths,residentstotalalldeaths,residentsweeklycovid19deaths,residentstotalcovid19deaths,numberofallbeds,totalnumberofoccupiedbeds,residentaccesstotestinginfacilit,laboratorytypeisstatehealthdept,laboratorytypeisprivatelab,laboratorytypeisother,staffweeklyconfirmedcovid19,stafftotalconfirmedcovid19,staffweeklysuspectedcovid19,stafftotalsuspectedcovid19,staffweeklycovid19deaths,stafftotalcovid19deaths,shortageofnursingstaff,shortageofclinicalstaff,shortageofaides,shortageofotherstaff,anycurrentsupplyofn95masks,oneweeksupplyofn95masks,anycurrentsupplyofsurgicalmasks,oneweeksupplyofsurgicalmasks,anycurrentsupplyofeyeprotection,oneweeksupplyofeyeprotection,anycurrentsupplyofgowns,oneweeksupplyofgowns,anycurrentsupplyofgloves,oneweeksupplyofgloves,anycurrentsupplyofhandsanitizer,oneweeksupplyofhandsanitizer,ventilatordependentunit,numberofventilatorsinfacility,numberofventilatorsinuseforcovid,anycurrentsupplyofventilatorsupp,oneweeksupplyofventilatorsupplie,totalresidentconfirmedcovid19cas,totalresidentcovid19deathsper100,totalresidentscovid19deathsasape,Building_ID,_ID,cleaned_address,phone,county_ssa,county_name,ownership,bedcert,restot,certification,inhosp,lbn,participation_date,ccrc_facil,sffstatus,abuse_icon,oldsurvey,chow_last_12mos,resfamcouncil,sprinkler_status,overall_rating,overall_rating_fn,survey_rating,survey_rating_fn,quality_rating,quality_rating_fn,ls_quality_rating,ls_quality_rating_fn,ss_quality_rating,ss_quality_rating_fn,staffing_rating,staffing_rating_fn,rn_staffing_rating,rn_staffing_rating_fn,staffing_flag,pt_staffing_flag,aidhrd,vochrd,rnhrd,totlichrd,tothrd,pthrd,cm_aide,cm_lpn,cm_rn,cm_total,adj_aide,adj_lpn,adj_rn,adj_total,weighted_all_cycles_score,incident_cnt,cmplnt_cnt,fine_cnt,fine_tot,payden_cnt,tot_penlty_cnt,State_Long,lat,lng,accuracy_check,Hand_Searched,building_match,building_match_convex,OSM_Building_Good,gisjoin,Links_Degree,Links_Strength,PrEigenVector1,degree_weeks_1_5,eigenvector_weeks_1_5,strength_weeks_1_5,weight_nghbr_degree_weeks_1_5,strength_days_weeks_1_5,degree_weeks_7_11,eigenvector_weeks_7_11,strength_weeks_7_11,weight_nghbr_degree_weeks_7_11,strength_days_weeks_7_11,degree_weeks_1_11,eigenvector_weeks_1_11,strength_weeks_1_11,weight_nghbr_degree_weeks_1_11,strength_days_weeks_1_11,FillIn,degree_cases,expsuminfectviol,count,shareblack,highblackshare,sharedual,highmedicaid,Unmatch,State_ResidentsPositive,May31_residents_deaths_state,May31_staff_pos_state,May31_staff_deaths_state,May31_datereport_state,May31_state,countyfipsstate,countyfipscounty,rstate,rcounty,FIPScode,codeurban2013,urban,forprofit,forprofit_ind,HasViolations,QualityAssuranceCheck,SurgicalMasks_OneWeekSupply,HandSanitizer_OneWeekSupply,HypArcSin_StateCases,CMS_ConfirmedPlusSuspected,HypArcSin_CMSCases,CMS_ConfirmedPlusSusp_01,State_ResidentsPositive_0_1,ratingdum1,ratingdum2,ratingdum3,ratingdum4,ratingdum5
1857,5/31/2020,06A088,LITTLE SISTERS OF THE POOR - MULLEN HOME,3629 W 29TH AVE,DENVER,CO,80211,Y,Y,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,42.0,37.0,Y,N,Y,N,0.0,0.0,0.0,0.0,0.0,0.0,N,N,N,N,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,N,NaN,NaN,NaN,NaN,0.0,0.0,NaN,Colorado_1926991,1889,"3629 W 29th Ave, Denver, CO 80211, USA",3034337221,150,Denver,Non profit - Corporation,42,40.500000,Medicaid,N,Legal Business Name Not Available,1975-05-01,N,NaN,N,N,N,Resident,Yes,5.0,NaN,5.0,NaN,3.0,NaN,3.0,NaN,NaN,2.0,5.0,NaN,5.0,NaN,NaN,NaN,3.39179,0.37251,1.28863,1.66114,5.05294,0.00000,1.69276,0.62319,0.27806,2.59401,4.16437,0.44947,1.74431,6.24633,8.000,0,0,0,0,0,0,Colorado,39.759281,-105.03612,ROOFTOP,0,0,0,1,G08003100003036,0.0,0.0,0.000000e+00,0,0.0,0,0.0,0,0,0.0,0,0.0,0,0,1.210000e-17,0,0.0,0,0,0,1.0,6.0

In [6]:
prisons_df = pd.read_csv(os.path.join(NURSE_CORRECTION_FOLDER,"facilities.csv"))
prisons_df = prisons_df[prisons_df["facility_state"]=="Colorado"]
#prisons_df["total_inmate_cases"] = prisons_df["total_inmate_cases"].fillna(prisons_df["total_inmate_deaths"])
#prisons_df["max_inmate_population_2020"] = prisons_df["max_inmate_population_2020"].fillna(prisons_df["total_inmate_cases"])
#prisons_df["latest_inmate_population"] = prisons_df["latest_inmate_population"].fillna(prisons_df["max_inmate_population_2020"])
prisons_df.head()

,nyt_id,facility_name,facility_type,facility_city,facility_county,facility_county_fips,facility_state,facility_lng,facility_lat,latest_inmate_population,max_inmate_population_2020,total_inmate_cases,total_inmate_deaths,total_officer_cases,total_officer_deaths,note
158,84EB1D78,Arkansas Valley Correctional Facility,State prison,Ordway,Crowley,8025,Colorado,-103.839378,38.189140,1012.0,NaN,961,4,99,0.0,NaN
159,A2A88844,Arrowhead Correctional Center,State prison,Cañon City,Fremont,8043,Colorado,-105.156106,38.434831,352.0,528.0,360,1,52,0.0,NaN
160,03079583,Bent County Correctional Facility,State prison,Las Animas,Bent,8011,Colorado,-103.205579,38.065192,1264.0,NaN,1091,2,95,2.0,NaN
161,5057D4F9,Buena Vista Correctional Facility,State prison,Buena Vista,Chaffee,8015,Colorado,-106.117628,38.822007,892.0,1129.0,523,1,89,0.0,NaN
162,5E8ED28B,Centennial Correctional Facility,State prison,Cañon City,Fremont,8043,Colorado,-105.160995,38.422731,700.0,886.0,151,0,102,0.0,NaN


In [7]:
nursing_cross_df[nursing_cross_df["Provider_ID"] == ("065001")]

,weekending,Provider_ID,providername,provideraddress,providercity,providerstate,providerzipcode,submitteddata,passedqualityassurancecheck,residentsweeklyadmissionscovid19,residentstotaladmissionscovid19,residentsweeklyconfirmedcovid19,CMS_ResidentsTotalConfirmed,residentsweeklysuspectedcovid19,residentstotalsuspectedcovid19,residentsweeklyalldeaths,residentstotalalldeaths,residentsweeklycovid19deaths,residentstotalcovid19deaths,numberofallbeds,totalnumberofoccupiedbeds,residentaccesstotestinginfacilit,laboratorytypeisstatehealthdept,laboratorytypeisprivatelab,laboratorytypeisother,staffweeklyconfirmedcovid19,stafftotalconfirmedcovid19,staffweeklysuspectedcovid19,stafftotalsuspectedcovid19,staffweeklycovid19deaths,stafftotalcovid19deaths,shortageofnursingstaff,shortageofclinicalstaff,shortageofaides,shortageofotherstaff,anycurrentsupplyofn95masks,oneweeksupplyofn95masks,anycurrentsupplyofsurgicalmasks,oneweeksupplyofsurgicalmasks,anycurrentsupplyofeyeprotection,oneweeksupplyofeyeprotection,anycurrentsupplyofgowns,oneweeksupplyofgowns,anycurrentsupplyofgloves,oneweeksupplyofgloves,anycurrentsupplyofhandsanitizer,oneweeksupplyofhandsanitizer,ventilatordependentunit,numberofventilatorsinfacility,numberofventilatorsinuseforcovid,anycurrentsupplyofventilatorsupp,oneweeksupplyofventilatorsupplie,totalresidentconfirmedcovid19cas,totalresidentcovid19deathsper100,totalresidentscovid19deathsasape,Building_ID,_ID,cleaned_address,phone,county_ssa,county_name,ownership,bedcert,restot,certification,inhosp,lbn,participation_date,ccrc_facil,sffstatus,abuse_icon,oldsurvey,chow_last_12mos,resfamcouncil,sprinkler_status,overall_rating,overall_rating_fn,survey_rating,survey_rating_fn,quality_rating,quality_rating_fn,ls_quality_rating,ls_quality_rating_fn,ss_quality_rating,ss_quality_rating_fn,staffing_rating,staffing_rating_fn,rn_staffing_rating,rn_staffing_rating_fn,staffing_flag,pt_staffing_flag,aidhrd,vochrd,rnhrd,totlichrd,tothrd,pthrd,cm_aide,cm_lpn,cm_rn,cm_total,adj_aide,adj_lpn,adj_rn,adj_total,weighted_all_cycles_score,incident_cnt,cmplnt_cnt,fine_cnt,fine_tot,payden_cnt,tot_penlty_cnt,State_Long,lat,lng,accuracy_check,Hand_Searched,building_match,building_match_convex,OSM_Building_Good,gisjoin,Links_Degree,Links_Strength,PrEigenVector1,degree_weeks_1_5,eigenvector_weeks_1_5,strength_weeks_1_5,weight_nghbr_degree_weeks_1_5,strength_days_weeks_1_5,degree_weeks_7_11,eigenvector_weeks_7_11,strength_weeks_7_11,weight_nghbr_degree_weeks_7_11,strength_days_weeks_7_11,degree_weeks_1_11,eigenvector_weeks_1_11,strength_weeks_1_11,weight_nghbr_degree_weeks_1_11,strength_days_weeks_1_11,FillIn,degree_cases,expsuminfectviol,count,shareblack,highblackshare,sharedual,highmedicaid,Unmatch,State_ResidentsPositive,May31_residents_deaths_state,May31_staff_pos_state,May31_staff_deaths_state,May31_datereport_state,May31_state,countyfipsstate,countyfipscounty,rstate,rcounty,FIPScode,codeurban2013,urban,forprofit,forprofit_ind,HasViolations,QualityAssuranceCheck,SurgicalMasks_OneWeekSupply,HandSanitizer_OneWeekSupply,HypArcSin_StateCases,CMS_ConfirmedPlusSuspected,HypArcSin_CMSCases,CMS_ConfirmedPlusSusp_01,State_ResidentsPositive_0_1,ratingdum1,ratingdum2,ratingdum3,ratingdum4,ratingdum5


In [8]:
nursing_CO_df = pd.merge(left=nursing_ts_df, right=nursing_cross_df, how="left", left_on="PROVNAME", right_on="providername")
#nursing_CO_df["numberofallbeds"] = nursing_CO_df["numberofallbeds"].fillna(nursing_CO_df["totalnumberofoccupiedbeds"])
nursing_CO_df

,Date,PROVNUM,PROVNAME,CITY,COUNTY,Case,ever_had_case,first_week,case_count_outbreak,case_count_CMS,Case_a,ever_had_case_a,first_week_a,homes_this,homes_last,homes_two,connections_this,connections_last,connections_two,homes_this_a,homes_last_a,homes_two_a,connections_this_a,connections_last_a,connections_two_a,OOS_homes,OOS_connections,weekending,Provider_ID,providername,provideraddress,providercity,providerstate,providerzipcode,submitteddata,passedqualityassurancecheck,residentsweeklyadmissionscovid19,residentstotaladmissionscovid19,residentsweeklyconfirmedcovid19,CMS_ResidentsTotalConfirmed,residentsweeklysuspectedcovid19,residentstotalsuspectedcovid19,residentsweeklyalldeaths,residentstotalalldeaths,residentsweeklycovid19deaths,residentstotalcovid19deaths,numberofallbeds,totalnumberofoccupiedbeds,residentaccesstotestinginfacilit,laboratorytypeisstatehealthdept,laboratorytypeisprivatelab,laboratorytypeisother,staffweeklyconfirmedcovid19,stafftotalconfirmedcovid19,staffweeklysuspectedcovid19,stafftotalsuspectedcovid19,staffweeklycovid19deaths,stafftotalcovid19deaths,shortageofnursingstaff,shortageofclinicalstaff,shortageofaides,shortageofotherstaff,anycurrentsupplyofn95masks,oneweeksupplyofn95masks,anycurrentsupplyofsurgicalmasks,oneweeksupplyofsurgicalmasks,anycurrentsupplyofeyeprotection,oneweeksupplyofeyeprotection,anycurrentsupplyofgowns,oneweeksupplyofgowns,anycurrentsupplyofgloves,oneweeksupplyofgloves,anycurrentsupplyofhandsanitizer,oneweeksupplyofhandsanitizer,ventilatordependentunit,numberofventilatorsinfacility,numberofventilatorsinuseforcovid,anycurrentsupplyofventilatorsupp,oneweeksupplyofventilatorsupplie,totalresidentconfirmedcovid19cas,totalresidentcovid19deathsper100,totalresidentscovid19deathsasape,Building_ID,_ID,cleaned_address,phone,county_ssa,county_name,ownership,bedcert,restot,certification,inhosp,lbn,participation_date,ccrc_facil,sffstatus,abuse_icon,oldsurvey,chow_last_12mos,resfamcouncil,sprinkler_status,overall_rating,overall_rating_fn,survey_rating,survey_rating_fn,quality_rating,quality_rating_fn,ls_quality_rating,ls_quality_rating_fn,ss_quality_rating,ss_quality_rating_fn,staffing_rating,staffing_rating_fn,rn_staffing_rating,rn_staffing_rating_fn,staffing_flag,pt_staffing_flag,aidhrd,vochrd,rnhrd,totlichrd,tothrd,pthrd,cm_aide,cm_lpn,cm_rn,cm_total,adj_aide,adj_lpn,adj_rn,adj_total,weighted_all_cycles_score,incident_cnt,cmplnt_cnt,fine_cnt,fine_tot,payden_cnt,tot_penlty_cnt,State_Long,lat,lng,accuracy_check,Hand_Searched,building_match,building_match_convex,OSM_Building_Good,gisjoin,Links_Degree,Links_Strength,PrEigenVector1,degree_weeks_1_5,eigenvector_weeks_1_5,strength_weeks_1_5,weight_nghbr_degree_weeks_1_5,strength_days_weeks_1_5,degree_weeks_7_11,eigenvector_weeks_7_11,strength_weeks_7_11,weight_nghbr_degree_weeks_7_11,strength_days_weeks_7_11,degree_weeks_1_11,eigenvector_weeks_1_11,strength_weeks_1_11,weight_nghbr_degree_weeks_1_11,strength_days_weeks_1_11,FillIn,degree_cases,expsuminfectviol,count,shareblack,highblackshare,sharedual,highmedicaid,Unmatch,State_ResidentsPositive,May31_residents_deaths_state,May31_staff_pos_state,May31_staff_deaths_state,May31_datereport_state,May31_state,countyfipsstate,countyfipscounty,rstate,rcounty,FIPScode,codeurban2013,urban,forprofit,forprofit_ind,HasViolations,QualityAssuranceCheck,SurgicalMasks_OneWeekSupply,HandSanitizer_OneWeekSupply,HypArcSin_StateCases,CMS_ConfirmedPlusSuspected,HypArcSin_CMSCases,CMS_ConfirmedPlusSusp_01,State_ResidentsPositive_0_1,ratingdum1,ratingdum2,ratingdum3,ratingdum4,ratingdum5
0,19apr2020 00:00:00,065001,LOWRY HILLS CARE AND REHABILITATION,AURORA,Arapahoe County,1,1,19apr2020 00:00:00,4,0,1,1,19apr2020 00:00:00,1,0,0,2,0,0,1,0,0,2,0,0,0,0,5/31/2020,65001,LOWRY HILLS CARE AND REHABILITATION,10201 E THIRD AVE,AURORA,CO,80010.0,Y,Y,0.0,2.0,0.0,7.0,1.0,16.0,0.0,4.0,0.0,3.0,102.0,62.0,Y,Y,Y,N,0.0,6.0,2.0,20.0,0.0,0.0,N,N,N,N,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,Y,N,NaN,NaN,NaN,NaN,112.90000,48.400002,42.900002,Colorado_607866,1958.0,"102

In [9]:
nursing_CO_df.columns

Index(['Date', 'PROVNUM', 'PROVNAME', 'CITY', 'COUNTY', 'Case',
       'ever_had_case', 'first_week', 'case_count_outbreak', 'case_count_CMS',
       ...
       'HypArcSin_StateCases', 'CMS_ConfirmedPlusSuspected',
       'HypArcSin_CMSCases', 'CMS_ConfirmedPlusSusp_01',
       'State_ResidentsPositive_0_1', 'ratingdum1', 'ratingdum2', 'ratingdum3',
       'ratingdum4', 'ratingdum5'],
      dtype='object', length=204)

I'm currently imputing total number of beds of each nursing home with total number of occupied beds (if the former is missing).

Same thing for prisons, I'm imputing:

latest_inmate_pop <- max_inmate_pop_2020 <- total_inmate_cases <- total_inmate_deaths

Even then, there will still be counties where there are no data for nursing homes or prisons. I will be treating those counties as having 0 beds and 0 inmates


### Generate Feature Data used for Matching Analysis

In [10]:
nursing_col_keep = ["COUNTY", 'PROVNUM', 'PROVNAME',"numberofallbeds"]
nursing_CO_use_df = nursing_CO_df[nursing_col_keep]
nursing_CO_use_df = pd.merge(left=county_fips_master_df[["fips","county_name"]],right=nursing_CO_use_df,how="left",left_on="county_name",right_on="COUNTY")
nursing_CO_use_df = nursing_CO_use_df.drop(["county_name"],axis=1)
nursing_CO_use_df = nursing_CO_use_df.drop_duplicates().dropna(subset="numberofallbeds")
nursing_CO_use_df

,fips,COUNTY,PROVNUM,PROVNAME,numberofallbeds
0,8001,Adams County,065108,"VILLAS AT SUNNY ACRES, THE",160.0
1,8001,Adams County,065120,CLEAR CREEK CARE CENTER,100.0
2,8001,Adams County,065193,ALPINE LIVING CENTER,101.0
3,8001,Adams County,065196,AVAMERE TRANSITIONAL CARE AND REHAB-MALLEY,162.0
4,8001,Adams County,065238,ELMS HAVEN CENTER,242.0
...,...,...,...,...,...
4343,8123,Weld County,065368,LIFE CARE CENTER OF GREELEY,104.0
4344,8123,Weld County,065397,"GRACE POINTE CONT CARE SR CAMPUS, SKILLED NURSING",53.0
4345,8123,Weld County,065410,COLUMBINE COMMONS HEALTH AND REHAB LLC,30.0
4518,8125,Yuma County,065263,YUMA LIFE CARE CENTER,32.0


In [11]:
nursing_CO_use_df["fips"].unique().shape

(44,)

In [12]:
prisons_col_use = ["nyt_id","facility_name","facility_county_fips","max_inmate_population_2020"]
prisons_use_df = prisons_df[prisons_col_use]
prisons_use_df.head()

,nyt_id,facility_name,facility_county_fips,max_inmate_population_2020
158,84EB1D78,Arkansas Valley Correctional Facility,8025,NaN
159,A2A88844,Arrowhead Correctional Center,8043,528.0
160,03079583,Bent County Correctional Facility,8011,NaN
161,5057D4F9,Buena Vista Correctional Facility,8015,1129.0
162,5E8ED28B,Centennial Correctional Facility,8043,886.0


### Get County Level Features

In [13]:
county_features_df = pd.read_csv(os.path.join(DATA_FOLDER, "county_features.csv"))
colorado_county_features_df = county_features_df[county_features_df["STATE"]=="COLORADO"]

colorado_county_features_df["FIPS"] = colorado_county_features_df["FIPS"].astype(int)
relevant_cols = ["E_TOTPOP", "E_AGE17", "E_AGE65", "E_POV"]
colorado_county_features_df[relevant_cols] = colorado_county_features_df[relevant_cols].astype(int)

for f in relevant_cols[1:]:
    colorado_county_features_df[f+"_PERCENTAGE"] = 100*colorado_county_features_df[f]/colorado_county_features_df["E_TOTPOP"]

features_to_keep = ["FIPS", "COUNTY", "STATE", "LAT", "LON", "AREA_SQMI", "E_TOTPOP", "E_AGE17", "E_AGE65", "E_POV"] + [f+"_PERCENTAGE" for f in relevant_cols[1:]]

colorado_county_features_df = colorado_county_features_df[features_to_keep]

colorado_county_features_df["E_TOTPOP_RANK"] = colorado_county_features_df["E_TOTPOP"].rank(ascending=False)

colorado_county_features_df

,FIPS,COUNTY,STATE,LAT,LON,AREA_SQMI,E_TOTPOP,E_AGE17,E_AGE65,E_POV,E_AGE17_PERCENTAGE,E_AGE65_PERCENTAGE,E_POV_PERCENTAGE,E_TOTPOP_RANK
244,8001,Adams,COLORADO,39.874325,-104.331872,1166.256942,497115,135444,49181,56588,27.246009,9.893284,11.383282,5.0
245,8003,Alamosa,COLORADO,37.568442,-105.788041,722.576715,16444,3932,2113,3635,23.911457,12.849672,22.105327,31.0
246,8005,Arapahoe,COLORADO,39.644554,-104.331706,797.930917,636671,153460,78627,56802,24.103501,12.349707,8.921719,3.0
247,8007,Archuleta,COLORADO,37.202395,-107.050863,1350.086257,12908,2209,3168,1370,17.113418,24.542919,10.613573,35.0
248,8009,Baca,COLORADO,37.309780,-102.543741,2554.991586,3563,739,923,661,20.740949,25.905136,18.551782,56.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
303,8117,Summit,COLORADO,39.621023,-106.137555,608.338717,30429,5031,3601,3075,16.533570,11.834106,10.105491,18.0
304,8119,Teller,COLORADO,38.869976,-105.187365,557.042574,24113,4368,4875,1980,18.114710,20.217310,8.211338,24.0
305,8121,Washington,COLORADO,39.965790,-103.209744,2518.082648,4840,1108,944,480,22.892562,19.504132,9.917355,51.0
306,8123,Weld,COLORADO,40.555961,-104.383666,3986.025890,295123,78590,34304,30574,26.629575,11.623628,10.359748,9.0


### Tally the total number of inmates and nusing home beds per county

In [14]:
beds_per_county_df = nursing_CO_use_df.groupby(["fips"])["numberofallbeds"].agg(num_beds='sum', count_homes='count').reset_index(level=0)
inmates_per_county_df = prisons_use_df.groupby("facility_county_fips")["max_inmate_population_2020"].agg(num_inmates='sum', count_facilities='count').reset_index(level=0)
feature_match_df = pd.merge(left=beds_per_county_df, right=inmates_per_county_df, left_on="fips", right_on="facility_county_fips", how="outer")
feature_match_df = feature_match_df.drop(["facility_county_fips"],axis=1)
feature_match_df = feature_match_df.fillna(0)
feature_match_df = feature_match_df.astype(np.int64)
feature_match_df = pd.merge(left=feature_match_df, right=colorado_county_features_df, how="inner",left_on="fips",right_on="FIPS")
# Write to output folder
#feature_match_df.to_csv(os.path.join(OUTPUT_FOLDER,"feature_match.csv"),index=False)
#feature_match_df_rank = feature_match_df.copy()
#feature_match_df_rank[["num_beds","num_inmates"]] = feature_match_df_rank[["num_beds","num_inmates"]].rank(method="dense",ascending=False)
#feature_match_df_rank

In [15]:
feature_match_df

,fips,num_beds,count_homes,num_inmates,count_facilities,FIPS,COUNTY,STATE,LAT,LON,AREA_SQMI,E_TOTPOP,E_AGE17,E_AGE65,E_POV,E_AGE17_PERCENTAGE,E_AGE65_PERCENTAGE,E_POV_PERCENTAGE,E_TOTPOP_RANK
0,8001,1671,14,0,0,8001,Adams,COLORADO,39.874325,-104.331872,1166.256942,497115,135444,49181,56588,27.246009,9.893284,11.383282,5.0
1,8003,123,2,0,0,8003,Alamosa,COLORADO,37.568442,-105.788041,722.576715,16444,3932,2113,3635,23.911457,12.849672,22.105327,31.0
2,8005,1996,18,0,0,8005,Arapahoe,COLORADO,39.644554,-104.331706,797.930917,636671,153460,78627,56802,24.103501,12.349707,8.921719,3.0
3,8007,60,1,0,0,8007,Archuleta,COLORADO,37.202395,-107.050863,1350.086257,12908,2209,3168,1370,17.113418,24.542919,10.613573,35.0
4,8009,86,2,0,0,8009,Baca,COLORADO,37.309780,-102.543741,2554.991586,3563,739,923,661,20.740949,25.905136,18.551782,56.0
5,8011,56,1,0,0,8011,Bent,COLORADO,37.931891,-103.077584,1512.846146,5809,894,1005,912,15.389912,17.300740,15.699776,48.0
6,8013,1078,11,0,0,8013,Boulder,COLORADO,40.094970,-105.397691,726.369908,321030,62925,42068,38546,19.600972,13.104071,12.006978,8.0
7,8014,200,1,0,0,8014,Broomfield,COLORADO,39.953593,-105.050787,33.003534,66120,15692,8499,3296,23.732607,12.853902,4.984876,12.0
8,8015,98,1,1129,1,8015,Chaffee,COLORADO,38.738223,-106.316683,1013.407978,19164,2979,4670,1807,15.544771,24.368608,9.429138,26.0
9,8017,38,1,0,0,8017,Cheyenne,COLORADO,38.835646,-102.601791,1778.275373,2039,692,245,228,33.938205,12.015694,11.181952,58.0


In [16]:
feature_match_df

,fips,num_beds,count_homes,num_inmates,count_facilities,FIPS,COUNTY,STATE,LAT,LON,AREA_SQMI,E_TOTPOP,E_AGE17,E_AGE65,E_POV,E_AGE17_PERCENTAGE,E_AGE65_PERCENTAGE,E_POV_PERCENTAGE,E_TOTPOP_RANK
0,8001,1671,14,0,0,8001,Adams,COLORADO,39.874325,-104.331872,1166.256942,497115,135444,49181,56588,27.246009,9.893284,11.383282,5.0
1,8003,123,2,0,0,8003,Alamosa,COLORADO,37.568442,-105.788041,722.576715,16444,3932,2113,3635,23.911457,12.849672,22.105327,31.0
2,8005,1996,18,0,0,8005,Arapahoe,COLORADO,39.644554,-104.331706,797.930917,636671,153460,78627,56802,24.103501,12.349707,8.921719,3.0
3,8007,60,1,0,0,8007,Archuleta,COLORADO,37.202395,-107.050863,1350.086257,12908,2209,3168,1370,17.113418,24.542919,10.613573,35.0
4,8009,86,2,0,0,8009,Baca,COLORADO,37.309780,-102.543741,2554.991586,3563,739,923,661,20.740949,25.905136,18.551782,56.0
5,8011,56,1,0,0,8011,Bent,COLORADO,37.931891,-103.077584,1512.846146,5809,894,1005,912,15.389912,17.300740,15.699776,48.0
6,8013,1078,11,0,0,8013,Boulder,COLORADO,40.094970,-105.397691,726.369908,321030,62925,42068,38546,19.600972,13.104071,12.006978,8.0
7,8014,200,1,0,0,8014,Broomfield,COLORADO,39.953593,-105.050787,33.003534,66120,15692,8499,3296,23.732607,12.853902,4.984876,12.0
8,8015,98,1,1129,1,8015,Chaffee,COLORADO,38.738223,-106.316683,1013.407978,19164,2979,4670,1807,15.544771,24.368608,9.429138,26.0
9,8017,38,1,0,0,8017,Cheyenne,COLORADO,38.835646,-102.601791,1778.275373,2039,692,245,228,33.938205,12.015694,11.181952,58.0


### Split Choices into CDPHE (x) and TLGRF (y)

In [17]:
feature_choice_cols = ["date.y","fips","county"]
# Merge CDPHE choice with feature_match_df
compare_choices_df = pd.read_csv(os.path.join(OUTPUT_FOLDER,"case_study_decision_space.csv"))

alg_feature_dict = {}

for method in ["exp","logistic"]:
    alg_choices_df = compare_choices_df[feature_choice_cols + ["chosen_by_CDPHE"]]
    alg_choices_df = alg_choices_df[alg_choices_df["chosen_by_CDPHE"] == 1]
    alg_choices_df = pd.merge(left=alg_choices_df,right=feature_match_df,how="inner",on="fips")
    alg_choices_df = alg_choices_df.fillna(0)
    alg_feature_dict["CDPHE_{}".format(method)] = alg_choices_df
    for n in range(1,10):
        TLGRF_N = "TLGRF{}".format(n)
        alg_choices_df = compare_choices_df[feature_choice_cols + ["chosen_by_{}_{}".format(TLGRF_N,method)]]
        alg_choices_df = alg_choices_df[alg_choices_df["chosen_by_{}_{}".format(TLGRF_N,method)] == 1]
        alg_choices_df = pd.merge(left=alg_choices_df,right=feature_match_df,how="inner",on="fips")
        alg_choices_df = alg_choices_df.fillna(0)
        alg_feature_dict[TLGRF_N + "_{}".format(method)] = alg_choices_df




In [18]:
alg_feature_dict

{'CDPHE_exp':         date.y  fips     county  chosen_by_CDPHE  num_beds  count_homes  \
 0   2020-04-08  8051   Gunnison                1        50            1   
 1   2020-07-22  8051   Gunnison                1        50            1   
 2   2021-01-20  8051   Gunnison                1        50            1   
 3   2021-04-25  8051   Gunnison                1        50            1   
 4   2021-06-15  8051   Gunnison                1        50            1   
 ..         ...   ...        ...              ...       ...          ...   
 78  2021-05-11  8021    Conejos                1        60            1   
 79  2020-12-21  8039     Elbert                1        30            1   
 80  2021-02-07  8017   Cheyenne                1        38            1   
 81  2021-06-23  8013    Boulder                1      1078           11   
 82  2021-06-27  8059  Jefferson                1      2308           23   
 
     num_inmates  count_facilities  FIPS     COUNTY     STATE        LAT 

In [19]:
alg_rankings_dict = {}
columns_to_use = ["num_beds","num_inmates","E_TOTPOP","E_TOTPOP_RANK"]
for alg in alg_feature_dict.keys():
    int_df = alg_feature_dict[alg][columns_to_use]
    int_df["num_beds_RANK"] = int_df["num_beds"].rank(method="dense",ascending=False)
    int_df["num_inmates_RANK"] = int_df["num_inmates"].rank(method="dense",ascending=False)
    
    alg_rankings_dict[alg] = int_df

In [20]:
alg_rankings_dict

{'CDPHE_exp':     num_beds  num_inmates  E_TOTPOP  E_TOTPOP_RANK  num_beds_RANK  \
 0         50            0     16537           30.0           22.0   
 1         50            0     16537           30.0           22.0   
 2         50            0     16537           30.0           22.0   
 3         50            0     16537           30.0           22.0   
 4         50            0     16537           30.0           22.0   
 ..       ...          ...       ...            ...            ...   
 78        60            0      8142           40.0           19.0   
 79        30            0     25162           22.0           27.0   
 80        38            0      2039           58.0           23.0   
 81      1078            0    321030            8.0            2.0   
 82      2308         1127    570427            4.0            1.0   
 
     num_inmates_RANK  
 0                6.0  
 1                6.0  
 2                6.0  
 3                6.0  
 4                6.0  
 

In [21]:
alg_mean_rankings_dict = {}
alg_median_rankings_dict = {}
for alg in alg_rankings_dict.keys():
    alg_mean_rankings_dict[alg] = alg_rankings_dict[alg].mean()
    alg_median_rankings_dict[alg] = alg_rankings_dict[alg].median()


In [22]:
alg_mean_rankings_dict

{'CDPHE_exp': num_beds              158.867470
 num_inmates           589.843373
 E_TOTPOP            33324.120482
 E_TOTPOP_RANK          30.397590
 num_beds_RANK          15.831325
 num_inmates_RANK        5.397590
 dtype: float64,
 'TLGRF1_exp': num_beds               75.811594
 num_inmates           176.115942
 E_TOTPOP            12595.463768
 E_TOTPOP_RANK          41.637681
 num_beds_RANK          12.971014
 num_inmates_RANK        3.797101
 dtype: float64,
 'TLGRF2_exp': num_beds               81.958904
 num_inmates           123.041096
 E_TOTPOP            15444.753425
 E_TOTPOP_RANK          37.205479
 num_beds_RANK          16.561644
 num_inmates_RANK        2.863014
 dtype: float64,
 'TLGRF3_exp': num_beds               91.500000
 num_inmates           154.040541
 E_TOTPOP            16694.878378
 E_TOTPOP_RANK          36.581081
 num_beds_RANK          14.743243
 num_inmates_RANK        2.945946
 dtype: float64,
 'TLGRF4_exp': num_beds               86.272727
 num_inmates 

In [23]:
alg_mean_rankings_df = pd.DataFrame()
alg_median_rankings_df = pd.DataFrame()
for alg in alg_mean_rankings_dict.keys():
    alg_mean_rankings_df = alg_mean_rankings_df.append(alg_mean_rankings_dict[alg], ignore_index=True)
    alg_median_rankings_df = alg_median_rankings_df.append(alg_median_rankings_dict[alg], ignore_index=True)
alg_mean_rankings_df["algorithm"] = alg_mean_rankings_dict.keys()
alg_median_rankings_df["algorithm"] = alg_median_rankings_dict.keys()

alg_mean_rankings_df = alg_mean_rankings_df[["algorithm"]+list(alg_mean_rankings_df.columns[:-1])]
alg_median_rankings_df = alg_median_rankings_df[["algorithm"]+list(alg_median_rankings_df.columns[:-1])]

confusion_CDPHE_TLGRF = pd.read_csv(os.path.join(OUTPUT_FOLDER,"confusion_CDPHE_TLGRF.csv"))
alg_mean_rankings_df = pd.merge(left=alg_mean_rankings_df,right=confusion_CDPHE_TLGRF[["algorithm","PPV","NPV"]],how="inner",on="algorithm")
alg_median_rankings_df = pd.merge(left=alg_median_rankings_df,right=confusion_CDPHE_TLGRF[["algorithm","PPV","NPV"]],how="inner",on="algorithm")


alg_mean_rankings_df.to_csv(os.path.join(OUTPUT_FOLDER,"alg_mean_rankings_df.csv"),index=False)
alg_median_rankings_df.to_csv(os.path.join(OUTPUT_FOLDER,"alg_median_rankings_df.csv"),index=False)

In [24]:
alg_mean_rankings_df

,algorithm,num_beds,num_inmates,E_TOTPOP,E_TOTPOP_RANK,num_beds_RANK,num_inmates_RANK,PPV,NPV
0,CDPHE_exp,158.867470,589.843373,33324.120482,30.397590,15.831325,5.397590,0.153846,0.950942
1,TLGRF1_exp,75.811594,176.115942,12595.463768,41.637681,12.971014,3.797101,0.136752,0.949950
2,TLGRF2_exp,81.958904,123.041096,15444.753425,37.205479,16.561644,2.863014,0.128205,0.949455
3,TLGRF3_exp,91.500000,154.040541,16694.878378,36.581081,14.743243,2.945946,0.102564,0.947968
4,TLGRF4_exp,86.272727,155.532468,15788.675325,35.935065,14.922078,3.909091,0.076923,0.946482
5,TLGRF5_exp,114.457831,382.795181,23337.373494,32.289157,18.036145,4.746988,0.059829,0.945491
6,TLGRF6_exp,98.529412,450.635294,21921.329412,33.000000,13.905882,4.658824,0.059829,0.945491
7,TLGRF7_exp,112.489583,157.510417,22279.125000,32.125000,12.614583,3.916667,0.051282,0.944995
8,TLGRF8_exp,121.376812,311.260870,19339.985507,33.188406,10.449275,3.855072,0.051282,0.944995
9,TLGRF9_exp,109.459770,276.954023,21170.126437,30.896552,14.609195,4.793103,0.025641,0.943508


In [25]:
alg_feature_dict["CDPHE_logistic"].head()

,date.y,fips,county,chosen_by_CDPHE,num_beds,count_homes,num_inmates,count_facilities,FIPS,COUNTY,STATE,LAT,LON,AREA_SQMI,E_TOTPOP,E_AGE17,E_AGE65,E_POV,E_AGE17_PERCENTAGE,E_AGE65_PERCENTAGE,E_POV_PERCENTAGE,E_TOTPOP_RANK
0,2020-04-08,8051,Gunnison,1,50,1,0,0,8051,Gunnison,COLORADO,38.670499,-107.05688,3239.140212,16537,2929,2044,2084,17.711798,12.360162,12.602044,30.0
1,2020-07-22,8051,Gunnison,1,50,1,0,0,8051,Gunnison,COLORADO,38.670499,-107.05688,3239.140212,16537,2929,2044,2084,17.711798,12.360162,12.602044,30.0
2,2021-01-20,8051,Gunnison,1,50,1,0,0,8051,Gunnison,COLORADO,38.670499,-107.05688,3239.140212,16537,2929,2044,2084,17.711798,12.360162,12.602044,30.0
3,2021-04-25,8051,Gunnison,1,50,1,0,0,8051,Gunnison,COLORADO,38.670499,-107.05688,3239.140212,16537,2929,2044,2084,17.711798,12.360162,12.602044,30.0
4,2021-06-15,8051,Gunnison,1,50,1,0,0,8051,Gunnison,COLORADO,38.670499,-107.05688,3239.140212,16537,2929,2044,2084,17.711798,12.360162,12.602044,30.0


In [26]:
alg_feature_dict["TLGRF1_logistic"].head()

,date.y,fips,county,chosen_by_TLGRF1_logistic,num_beds,count_homes,num_inmates,count_facilities,FIPS,COUNTY,STATE,LAT,LON,AREA_SQMI,E_TOTPOP,E_AGE17,E_AGE65,E_POV,E_AGE17_PERCENTAGE,E_AGE65_PERCENTAGE,E_POV_PERCENTAGE,E_TOTPOP_RANK
0,2020-04-08,8037,Eagle,1,0,1,0,0,8037,Eagle,COLORADO,39.626992,-106.695169,1684.470957,54357,12118,5482,3793,22.293357,10.085178,6.977942,15.0
1,2020-07-15,8037,Eagle,1,0,1,0,0,8037,Eagle,COLORADO,39.626992,-106.695169,1684.470957,54357,12118,5482,3793,22.293357,10.085178,6.977942,15.0
2,2021-01-18,8037,Eagle,1,0,1,0,0,8037,Eagle,COLORADO,39.626992,-106.695169,1684.470957,54357,12118,5482,3793,22.293357,10.085178,6.977942,15.0
3,2021-01-26,8037,Eagle,1,0,1,0,0,8037,Eagle,COLORADO,39.626992,-106.695169,1684.470957,54357,12118,5482,3793,22.293357,10.085178,6.977942,15.0
4,2020-04-20,8029,Delta,1,185,3,0,0,8029,Delta,COLORADO,38.861595,-107.864892,1142.126464,30346,6176,7499,5145,20.351941,24.711659,16.954459,19.0


In [27]:
feature_cols = ["num_beds","num_inmates","E_TOTPOP"]
CDPHE_choice = alg_feature_dict["CDPHE_logistic"]
TLGRF1_logistic_choice = alg_feature_dict["TLGRF1_logistic"]
#print("1 sided t-test of CDPHE against TLGRF1_logistic")
for feature in feature_cols:
    tstat, pvalue = scipy.stats.ttest_ind(a=CDPHE_choice[feature], b=TLGRF1_logistic_choice[feature],equal_var=False)
    print("For {}, t-statistic is {}, pvalue is {}".format(feature,tstat,pvalue))

For num_beds, t-statistic is 0.9345124664521997, pvalue is 0.3521515534519438
For num_inmates, t-statistic is 0.3339051388372595, pvalue is 0.7388853661726668
For E_TOTPOP, t-statistic is 0.30826805328597223, pvalue is 0.7585173361515981


In [28]:
CDPHE_choice["num_beds"].mean(), CDPHE_choice["num_beds"].var()**0.5

(158.86746987951807, 284.7508868948837)

In [29]:
rankings_df.mean()

rankings_data = {"Mean Total Nursing Home Beds": [rankings_df.mean()["total_beds_x"],rankings_df.mean()["total_beds_y"]]}
rankings_data["Mean Total Inmate Capacity"] = [rankings_df.mean()["total_inmates_x"],rankings_df.mean()["total_inmates_y"]]
rankings_data["Mean Total County Population"] = [rankings_df.mean()["POP_RANK_x"],rankings_df.mean()["POP_RANK_y"]]

rankings_table = pd.DataFrame.from_dict(rankings_data, orient="index")
rankings_table = rankings_table.rename(columns={0:"CDPHE", 1:"TLGRF"})
rankings_table.index.rename(name="Rank",inplace=True)
# Write to CSV
rankings_table.to_csv(os.path.join(OUTPUT_FOLDER,"rankings_table.csv"))
# Write to xtable
with open(os.path.join(OUTPUT_FOLDER,"rankings_table.tex"),"w") as tf:
    tf.write(rankings_table.to_latex())

rankings_table

NameError: name 'rankings_df' is not defined

In [ ]:
rankings_table.to_latex()

In [ ]:
CDPHE_not_TLGRF = list(set(CDPHE_choice_df["fips"]) - set(CDPHE_choice_df["fips"]).intersection(set(TLGRF_choice_df["fips"])))
CDPHE_choice_df[CDPHE_choice_df["fips"].isin(CDPHE_not_TLGRF)]

In [ ]:
for feature in X_vector_cols:
    Y_vector = TLGRF_choice_df[feature] - CDPHE_choice_df[feature]
    Y_bar = Y_vector.mean
    cov = EmpiricalCovariance().fit(np.array([Y_vector]))
    Tsquare = abs(len(Y_vector)*(Ybar.T@np.linalg.inv(cov.covariance_)@Ybar))
    print("For feature {}, T2 statistic is {}".format(feature,Tsquare))


In [ ]:
cov = EmpiricalCovariance().fit(Y_vector)
Ybar = Y_vector.mean()
cov.covariance_, Ybar

In [ ]:
Tsquare = len(Y_vector)*(Ybar.T@np.linalg.inv(cov.covariance_)@Ybar)
abs(Tsquare)

In [ ]:
cov.mahalanobis(Y_vector)